# Pratica — Lezione 7: Calcolo su GPU con PyTorch

Curso **Deep Learning con Python y PyTorch desde cero** — DIH DATAlife

En esta práctica vas a resolver dos ejercicios:

1. Escribir una función que mueva un modelo a un `device` y compruebe que lo hizo correctamente
2. Medir el coste de mover un lote grande de datos al `device` en cada iteración frente a moverlo una sola vez

Intenta completar cada bloque marcado con `# TODO:` antes de mirar la solución. Este notebook funciona igual con o sin GPU (Graphics Processing Unit): en CPU (Central Processing Unit) los tiempos del ejercicio 2 serán parecidos entre sí, porque mover un tensor de CPU a CPU no tiene coste real. Si lo ejecutas en Google Colab con **Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU** activado, verás una diferencia mucho más clara.

In [ ]:
import torch
import torch.nn as nn
import time

print("Versión de PyTorch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo detectado:", device)

## Esercizio 1 — Sposta un modello al device correttamente

**Enunciado**: escribe una función `mover_modelo_a_device(modelo, device)` que mueva `modelo` al `device` indicado y compruebe con un `assert` que sus parámetros quedaron realmente allí, antes de devolver el modelo movido.

**Pistas**:

- `.to(device)` ya mueve el modelo "in place", pero reasignar el resultado (`modelo = modelo.to(device)`) es una buena práctica: deja claro qué variable contiene el modelo ya movido
- Para comprobar dónde están los parámetros de un modelo, mira el device del primero de ellos: `next(modelo.parameters()).device`
- Compara con `.type` (por ejemplo `next(modelo.parameters()).device.type == torch.device(device).type`) en lugar de comparar el objeto `device` completo: un `torch.device("cuda")` sin índice y un `torch.device("cuda:0")` no siempre se consideran iguales con `==`, aunque representen la misma GPU

In [ ]:
class ModeloEjemplo(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa = nn.Linear(10, 2)

    def forward(self, x):
        return self.capa(x)


def mover_modelo_a_device(modelo, device):
    """Mueve `modelo` a `device` y comprueba que sus parámetros quedaron allí."""
    # TODO: mueve el modelo al device indicado, reasignando el resultado a la variable `modelo`
    # TODO: comprueba con un assert que el device de next(modelo.parameters()) tiene el mismo
    #       .type que torch.device(device); si no coincide, lanza un mensaje de error útil
    # TODO: devuelve el modelo ya movido
    pass


modelo_prueba = ModeloEjemplo()
# TODO: llama a mover_modelo_a_device(modelo_prueba, device) y guarda el resultado en modelo_prueba
# TODO: imprime el device de next(modelo_prueba.parameters()) para comprobarlo a simple vista

<details>
<summary>💡 Vedi soluzione</summary>

```python
def mover_modelo_a_device(modelo, device):
    """Mueve `modelo` a `device` y comprueba que sus parámetros quedaron allí."""
    modelo = modelo.to(device)
    device_real = next(modelo.parameters()).device
    device_esperado = torch.device(device)
    assert device_real.type == device_esperado.type, (
        f"El modelo quedó en {device_real}, no en {device_esperado}"
    )
    return modelo


modelo_prueba = ModeloEjemplo()
modelo_prueba = mover_modelo_a_device(modelo_prueba, device)
print("Device de los parámetros tras moverlo:", next(modelo_prueba.parameters()).device)
```
</details>

## Esercizio 2 — Il costo di spostare i dati al device

**Enunciado**: crea un lote grande de datos sintéticos y un modelo lineal simple ya movido al `device`. Compara el tiempo de hacer 50 forward pass moviendo el lote al `device` en cada repetición, frente al tiempo de moverlo una sola vez y repetir solo el forward. El objetivo es razonar sobre el coste real de la transferencia de datos entre la CPU y la GPU, no sobre la precisión del modelo (por eso no hace falta entrenar nada, solo forward pass dentro de `torch.no_grad()`).

**Pistas**:

- Usa `torch.no_grad()` alrededor de ambos bucles: solo mides el forward, no necesitas construir el grafo de gradientes
- En la primera función, la línea `lote.to(device)` va **dentro** del bucle de repeticiones
- En la segunda función, la línea `lote.to(device)` va **una sola vez, antes** del bucle
- Mide el tiempo total de cada función completa con `time.time()`, no el de cada repetición por separado

In [ ]:
modelo_lineal = nn.Linear(1000, 10).to(device)
lote_grande = torch.randn(8192, 1000)   # nace en CPU, como cualquier lote de un DataLoader
repeticiones = 50


def forward_moviendo_cada_vez(lote, modelo, device, repeticiones):
    """Mueve el lote al device en cada repetición, justo antes del forward."""
    # TODO: dentro de un bloque "with torch.no_grad():", recorre range(repeticiones)
    # TODO: en cada iteración, mueve `lote` al device y pásalo por `modelo`
    pass


def forward_moviendo_una_vez(lote, modelo, device, repeticiones):
    """Mueve el lote al device una sola vez y repite solo el forward."""
    # TODO: mueve `lote` al device UNA vez, fuera del bucle
    # TODO: dentro de un bloque "with torch.no_grad():", recorre range(repeticiones)
    #       y pasa el lote (ya en el device) por `modelo` en cada iteración
    pass


# TODO: mide con time.time() el tiempo de forward_moviendo_cada_vez(lote_grande, modelo_lineal, device, repeticiones)
# TODO: mide con time.time() el tiempo de forward_moviendo_una_vez(lote_grande, modelo_lineal, device, repeticiones)
# TODO: imprime ambos tiempos y la diferencia entre ellos

<details>
<summary>💡 Vedi soluzione</summary>

```python
def forward_moviendo_cada_vez(lote, modelo, device, repeticiones):
    """Mueve el lote al device en cada repetición, justo antes del forward."""
    with torch.no_grad():
        for _ in range(repeticiones):
            lote_en_device = lote.to(device)
            _ = modelo(lote_en_device)


def forward_moviendo_una_vez(lote, modelo, device, repeticiones):
    """Mueve el lote al device una sola vez y repite solo el forward."""
    lote_en_device = lote.to(device)
    with torch.no_grad():
        for _ in range(repeticiones):
            _ = modelo(lote_en_device)


inicio = time.time()
forward_moviendo_cada_vez(lote_grande, modelo_lineal, device, repeticiones)
tiempo_cada_vez = time.time() - inicio

inicio = time.time()
forward_moviendo_una_vez(lote_grande, modelo_lineal, device, repeticiones)
tiempo_una_vez = time.time() - inicio

print(f"Moviendo el lote en cada iteración: {tiempo_cada_vez:.4f} segundos")
print(f"Moviendo el lote una sola vez:      {tiempo_una_vez:.4f} segundos")
print(f"Diferencia: {tiempo_cada_vez - tiempo_una_vez:.4f} segundos")
```
</details>

## Per terminare

Su CPU, la differenza tra le due funzioni dell'esercizio 2 sarà piccola, perché spostare un tensore dalla CPU alla CPU stessa non ha un costo reale di trasferimento. Se ripeti questo stesso esercizio su Google Colab con una GPU attivata, vedrai che `forward_moviendo_una_vez` è notevolmente più rapida di `forward_moviendo_cada_vez`: ogni chiamata a `.to(device)` quando la destinazione è una GPU implica un viaggio reale di dati attraverso il bus che collega la memoria della CPU e la memoria della GPU, e quel viaggio ha un costo che si sente molto se lo ripeti senza necessità. È la stessa ragione per cui, nel ciclo di addestramento del notebook di teoria, il modello viene spostato sul device una sola volta fuori dal ciclo, mentre solo i lotti di dati —che cambiano a ogni iterazione— vengono spostati al suo interno.